# Import lib

In [1]:
import torch
import torch.nn as nn

In [2]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(2, 1)

    def forward(self, x):
        return self.linear(x)

In [14]:
torch.manual_seed(42)
x = torch.randn(10, 1)   # 生成10个标准高斯(正态)随机数

w0 = torch.tensor([2.0, 1.0]) # [w, b] = [2,1]
x_tilde = torch.cat([x, torch.ones_like(x)], dim=1)
y = x_tilde @ w0.unsqueeze(1)

# 阈值判断：y>=1 →1，else 0
binary_y = torch.where(y >= 1, torch.tensor(1.0), torch.tensor(0.0))

print("x:\n", x)
print("y = 2*x +1:\n", y)
print("binary_y:\n", binary_y)

x:
 tensor([[ 0.3367],
        [ 0.1288],
        [ 0.2345],
        [ 0.2303],
        [-1.1229],
        [-0.1863],
        [ 2.2082],
        [-0.6380],
        [ 0.4617],
        [ 0.2674]])
y = 2*x +1:
 tensor([[ 1.6734],
        [ 1.2576],
        [ 1.4689],
        [ 1.4607],
        [-1.2457],
        [ 0.6273],
        [ 5.4164],
        [-0.2760],
        [ 1.9233],
        [ 1.5347]])
binary_y:
 tensor([[1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [0.],
        [1.],
        [0.],
        [1.],
        [1.]])


In [13]:
model = MyModel()
# ===== 保存训练前原始参数 =====
w_before = model.linear.weight.clone()
b_before = model.linear.bias.clone()
print("\n===== 训练前参数 =====")
print("Weight(w):", w_before)
print("Bias(b):", b_before)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(10):
	y_pred = model(x_tilde) # 1. forward
	loss = loss_fn(y_pred, binary_y) # 2. 算 Loss
	optimizer.zero_grad() # 3. 清空旧梯度
	loss.backward() # 4. 反向传播
	optimizer.step() # 5. 更新参数
	
	if epoch % 2 ==0:
		print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

w_after = model.linear.weight
b_after = model.linear.bias
print("\n===== 训练后参数 =====")
print("Weight(w):", w_after)
print("Bias(b):", b_after)

# ===== 计算参数差值 =====
delta_w = w_after - w_before
delta_b = b_after - b_before
print("\n===== 参数变化量 =====")
print("Weight delta: ", delta_w)
print("Bias delta: ", delta_b)
print("Weight abs max change: ", torch.max(torch.abs(delta_w)).item())
print("Bias abs change: ", torch.abs(delta_b).item())


===== 训练前参数 =====
Weight(w): tensor([[-0.2230,  0.1900]], grad_fn=<CloneBackward0>)
Bias(b): tensor([-0.1918], grad_fn=<CloneBackward0>)
Epoch 0, Loss: 0.7886
Epoch 2, Loss: 0.7875
Epoch 4, Loss: 0.7863
Epoch 6, Loss: 0.7852
Epoch 8, Loss: 0.7840

===== 训练后参数 =====
Weight(w): Parameter containing:
tensor([[-0.2130,  0.2000]], requires_grad=True)
Bias(b): Parameter containing:
tensor([-0.1818], requires_grad=True)

===== 参数变化量 =====
Weight delta:  tensor([[0.0100, 0.0100]], grad_fn=<SubBackward0>)
Bias delta:  tensor([0.0100], grad_fn=<SubBackward0>)
Weight abs max change:  0.009997561573982239
Bias abs change:  0.009969621896743774


In [4]:
class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 5)
        self.fc2 = nn.Linear(5, 2)

    def forward(self, x):
        x = self.fc1(x)
        return self.fc2(x)